# Распределение товаров по торговым точкам через роевой интеллект

## Библиотеки

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor  # для многопоточности

## Эвристики

Влияние продаж

In [2]:
sales_factor = 2

Влияние загруженности

In [3]:
load_factor = 10

Сколько итераций считать

In [4]:
iterations_count = 100

Ранняя остановка. Если истина, то тогда нужно указать величину на которую отличаются предыдущее лучшее решение и нынешнее

In [5]:
early_stopping = False
epsilon = 10**-4

Константа выделения феромонов

In [6]:
pheromone_quantity = 1

Испарение феромонов

In [7]:
pheromone_loss = 0.3

Влияние феромонов

In [8]:
pheromone_factor = 0.5

Влияние эвристик

In [9]:
heuristic_factor = 1

Число муравьёв

In [10]:
ant_count = 10

## Подгрузка датасета

In [11]:
col_vol = "Объём товара в закрывающемся магазине (V)"
col_maxcap = "Максимальная вместимость товара в магазине (M)"
col_item = "Код товара"
col_store = "Магазин"
col_status = "Ассортимент Статус"
col_remain = "Остаток товара в магазине (R)"
col_sales = "Продажи товара в магазине (S)"
col_measure = "Единица измерения"
col_left = "остаток_робот"
col_pher = "Феромоны"
col_dist = "Распределено"
col_weight = "Вес"
col_update = "Локальное обновление феромонов"

In [12]:
items = pd.read_csv(r"./filtered_datasets/items_closing_store.csv", sep=";")
items = items.set_index([col_item, col_store])  # для быстрой итерации
items

Объём товара в закрывающемся магазине (V)  \
Код товара Магазин                                              
226249     54                                            10.0   
           56                                            10.0   
           57                                            10.0   
           60                                            10.0   
           61                                            10.0   
...                                                       ...   
239388     95                                            26.0   
239397     38                                            12.0   
239398     38                                            13.0   
239515     38                                            42.0   
239516     38                                            42.0   

                    Продажи товара в магазине (S)  \
Код товара Магазин                                  
226249     54                            1.379157   
           56                            0.943567   
           57                            0.857143   
           60                            0.406742   
           61                            0.713024   
...                                           ...   
239388     95                            1.500000   
239397     38                            1.116279   
239398     38                            2.023256   
239515     38                            1.796296   
239516     38                            1.740741   

                    Максимальная вместимость товара в магазине (M)  \
Код товара Магазин                                                   
226249     54                                                    8   
           56                                                    8   
           57                                                    8   
           60                                                    8   
           61                                                    8   
...                                                            ...   
239388     95                                                   25   
239397     38                                                   24   
239398     38                                                   24   
239515     38                                                   30   
239516     38                                                   24   

                    Остаток товара в магазине (R)  
Код товара Магазин                                 
226249     54                                16.0  
           56                                22.0  
           57                                19.0  
           60                                28.0  
           61                                22.0  
...                                           ...  
239388     95                                20.0  
239397     38                                21.0  
239398     38                                45.0  
239515     38                                35.0  
239516     38                                28.0  

[413689 rows x 4 columns]

Со всем датасетом работать очень медленно. Гораздо лучше будет сделать несколько матриц

$i$ - товар, $i ∈ [1, N]$, $N$ - кол-во товаров

$j$ - магазин, $j ∈ [1, M]$, $M$ - кол-во магазинов

*closing_volume* - вектор размерности $N×1$, показывает кол-во товара $i$ в закрывающемся магазине

*sales* - матрица размерности $N×M$, показывает продажи товара $i$ в магазине $j$

*maxcap* - матрица размерности $N×M$, показывает вместимость на полках товара $i$ в магазине $j$

*remains* - матрица размерности $N×M$, показывает остатки товара $i$ в магазине $j$

*pheromones* - матрица размерности $N×M$, показывает феромоны для распределения товара $i$ в магазин $j$

Если товар $i$ не числится в магазине $j$, то тогда соответственные значения $(i, j)$ во всех матрицах равны 0

In [13]:
items_id = items.index.get_level_values(0).unique().values
N = items_id.shape[0]
N

5366

In [14]:
stores_id = items.index.get_level_values(1).unique().values
M = stores_id.shape[0]
M

98

In [15]:
closing_volume = np.zeros(N, dtype=np.int64)
sales = np.zeros((N, M), dtype=np.float64)
maxcap = np.zeros((N, M), dtype=np.int64)
remains = np.zeros((N, M), dtype=np.int64)
pheromones = np.zeros((N, M), dtype=np.float64)

for i in range(N):   
    for j in range(M):
        try:
            df = items.loc[(items_id[i], stores_id[j])]
            closing_volume[i] = df[col_vol]
            sales[i][j] = df[col_sales]
            maxcap[i][j] = df[col_maxcap]
            remains[i][j] = df[col_remain]
            pheromones[i][j] = 1

        except KeyError:
            continue

## Муравьиный алгоритм

### Логика движения муравья

Сначала фиксируется товар, потом для него ищется распределение по магазинам, опираясь на случайный выбор по вероятности. Муравей получает на вход пустое решение, но с феромонами. По ходу работы он его меняет, добавляет обновление феромонов и возвращает оценку

In [13]:
def ant_distribution(ant_items):
    print("Ant init")
    
    ant_items[col_dist] = 0
    
    for item_id, df in ant_items.groupby(level=0):
        available_stores = df.index.get_level_values(1)
        
        w, s, r, m = df[col_pher].values, df[col_sales].values, df[col_remain].values, df[col_maxcap].values  # из формулы для эвристики
    
        weights = w ** pheromone_factor * (s * r / (1 + r) * m / (m + r)) ** heuristic_factor  # начальные веса
    
        weights_sum = np.sum(weights)  # сумма всех весов, считается полностью только один раз для оптимизации
    
        zero_sales = False
    
        if weights_sum == 0.0:  # на случай, если продажи товара по всем магазинам нулевые, то выбираем исходя только из загруженности
            weights = m / (m + r)
            weights_sum = np.sum(weights)
            zero_sales = True
        
        item_count = df[col_vol].iloc[0]  # пока не кончатся товары для распределения
    
        while item_count > 0:
            selected_store = np.random.choice(available_stores, p=weights / weights_sum)  # выбираем магазин случайно по вероятности
            i = np.where(available_stores == selected_store)
            
            ant_items.loc[(item_id, selected_store), col_dist] += 1  # распределили 1 штуку товара
            c = ant_items.loc[(item_id, selected_store), col_dist]
    
            weights_sum -= weights[i]  # вычли из общей суммы старый вес, чтобы не пересчитывать
    
            if not zero_sales:  # пересчитываем вес для этой пары товар-магазин
                weights[i] = w[i] ** pheromone_factor * (s[i] * (c + r[i]) / (1 + r[i]) * m[i] / (m[i] + c + r[i])) ** heuristic_factor
            else:  # если продажи были нулевые по всем магазинам
                weights[i] = m[i] / (m[i] + c + r[i])
    
            weights_sum += weights[i]  # добавили к общей сумме новый вес, чтобы не пересчитывать
    
            item_count -= 1
                
    c, s, r, m = ant_items[col_dist].values, ant_items[col_sales].values, ant_items[col_remain].values, ant_items[col_maxcap].values
    rating = np.sum((c + r) * (sales_factor * s / (1 + r) - load_factor / m))  # оценка решения

    ant_items[col_update] = 0.0
    ant_items.loc[ant_items[col_dist] > 0, col_update] = pheromone_quantity * rating  # считаем обновление феромонов

    print("Ant finish")

    return ant_items[col_update].values, rating, ant_items[col_dist]

### Распараллеливание

Чтобы было быстрее считать, алгоритм распараллеливает муравьёв, каждый считает своё решение, а затем всё обновляется

(5366,)

In [ ]:
W = aco_items[col_pher].values
S = aco_items[col_sales].values
R = aco_items[col_remain].values
M = aco_items[col_maxcap].values

In [28]:
V = grouped.first()[col_vol].values
V

Код товара
226249     10.0
234761      3.0
237251      3.0
217430     12.0
230261     21.0
          ...  
233129    106.0
233133     61.0
233135     88.0
234167     40.0
239221     17.0
Name: Объём товара в закрывающемся магазине (V), Length: 5366, dtype: float64

In [14]:
def ant_colony_optimization(items):
    V = items.groupby(level=0).first()
    
    aco_items[col_pher] = 1.0  # изначально все феромоны равны 1

    previous_best_rating = None
    best_rating, best_soluion = None, None

    for i in range(iterations_count):
        print(f"Iteration {i+1} out of {iterations_count}")
        pheromone_updates, ratings, solutions = [], [], []

        with ProcessPoolExecutor() as executor:
            futures = [executor.submit(ant_distribution, aco_items.copy()) for _ in range(ant_count)]
    
            for future in futures:
                result = future.result()
                pheromone_updates.append(result[0])
                ratings.append(result[1])
                solutions.append(result[2])

        print("Updating pheromones")
        aco_items[col_pher] = (1 - pheromone_loss) * aco_items[col_pher].values + np.sum(pheromone_updates, axis=0)  # глобальное обновление феромонов
        max_index = np.argmax(ratings)

        current_best_rating = ratings[max_index]
        current_best_solution = solutions[max_index]

        print("Best rating for current iteration:", current_best_rating)

        if (best_rating is None and best_solution is None) or (current_best_rating > best_rating):
            best_rating = current_best_rating
            best_solution = current_best_solution

        print("Total best rating:", best_rating)

        if early_stopping:
            if (previous_best_rating is None) or (np.abs(previous_best_rating - current_best_rating) >= epsilon):
                previous_best_rating = current_best_rating

            else:
                print("Early stopping")
                break

        print()

    return best_rating, best_solution        

In [ ]:
best_rating, best_solution = ant_colony_optimization(items)

Iteration 1 out of 100
Ant init
Ant init
Ant init
Ant init
Ant init
Ant init


In [ ]:
best_rating

In [ ]:
best_solution